## **Example usage of PhaseNet model for five-class classification**

In [ ]:
!pip install seisbench

In [ ]:
try:
    import obspy

    obspy.read()
except TypeError:
    # Needs to restart the runtime once, because obspy only works properly after restart.
    print(
        "Stopping RUNTIME. If you run this code for the first time, this is expected. Colaboratory will restart automatically. Please run again."
    )
    exit()

## Imports

In [ ]:
import seisbench.models as sbm
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

## Read model file

In [ ]:
model = sbm.PhaseNet(classes=5)
model.load_state_dict(torch.load('./model/model_epoch_143.pt', map_location=torch.device('cpu')))
model = model.eval()
print(model)

## Read sample data (in case of obspy stream)

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

client = Client("GFZ")

t = UTCDateTime("2007/01/02 05:48:50")
stream = client.get_waveforms(
    network="CX",
    station="PB01",
    location="*",
    channel="HH?",
    starttime=t - 9.99,
    endtime=t + 50,
)

## Preprocessing the waveform

In [ ]:
def standardize(waves, mode='peak'):
    waves -= np.mean(waves, axis=-1, keepdims=True)
    denominator = None
    if mode=='peak':
        denominator = np.max(np.abs(waves), axis=-1, keepdims=True)
    elif mode=='std':
        denominator = np.sqrt(np.square(waves).mean(axis=-1))
        shape = denominator.shape
        denominator = denominator.reshape(shape[0], shape[1], 1)
    denominator[denominator == 0] = 1

    return waves / denominator

In [ ]:
wave_pre = np.vstack([stream[0].data, stream[1].data, stream[2].data])
wave_pre = signal.detrend(wave_pre, type='constant')
wave_pre = wave_pre.reshape(1,3,6000)
wave_pre = standardize(wave_pre, mode='std')

wave_fig = wave_pre[0]
fig = plt.figure(figsize=(15, 5))
ax = fig.add_subplot(111)
for i in range(3):
    ax.plot(np.arange(0,6000), wave_fig[i])
ax.legend()

## Prediction

In [ ]:
wave_torch = torch.from_numpy(wave_pre).float()
res = model(wave_torch) #out :torch.Size([1, 3, 3000])
res = res.to('cpu').detach().numpy().copy() #torch tensor->numpy

## Plot prediction result

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 6))
for i in range(3):
    axes[0].plot(np.arange(0,6000), wave_fig[i])
axes[0].set_title("Waveform components")

res_fig = res[0]
for i in range(5):
    axes[1].plot(np.arange(0,6000), res_fig[i])
axes[1].set_title("Five-class probabilities")
axes[1].legend(["OP", "LP", "LS", "OS", "N"], loc="upper right")

In [ ]:
res_opos = np.vstack([res[0][0], res[0][3]])

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 6))
for i in range(3):
    axes[0].plot(np.arange(0,6000), wave_fig[i])
axes[0].set_title("Waveform components")

for i in range(2):
    axes[1].plot(np.arange(0,6000), res_opos[i])
axes[1].set_title("OP & OS probabilities")
axes[1].legend(["OP", "OS"], loc="upper right")

In [ ]:
res_lpls = np.vstack([res[0][1], res[0][2]])

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 6))
for i in range(3):
    axes[0].plot(np.arange(0,6000), wave_fig[i])
axes[0].set_title("Waveform components")

for i in range(2):
    axes[1].plot(np.arange(0,6000), res_lpls[i])
axes[1].set_title("LP & LS probabilities")
axes[1].legend(["LP", "LS"], loc="upper right")